# Credit decision explainer decision-tree rules

---
### 1. Imports and config

In [ ]:
import re
import time
import subprocess
import numpy as np
import requests

OLLAMA_HOST = "http://localhost:11434"
MODEL_NAME = "phi4-mini-reasoning:latest"

---
### 2. Start Ollama

In [ ]:
def start_ollama(wait_sec=30):
    try:
        requests.get(OLLAMA_HOST, timeout=2)
        print("Ollama is already running")
        return
    except requests.RequestException:
        pass
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(wait_sec):
        time.sleep(1)
        try:
            requests.get(OLLAMA_HOST, timeout=2)
            print("Ollama started")
            return
        except requests.RequestException:
            pass
    raise RuntimeError("Ollama did not start - try running 'ollama serve' in a terminal.")
start_ollama()

Ollama is already running


---
### 3. Decision-tree rules

In [ ]:
def decide(dpd, dbr, housing, age):
    dpd_c = np.where(dpd >= 1, 1, 0)                                    # 0 = none/missing, 1 = one or more
    dbr_c = np.where(dbr <= 0.10, 0, np.where(dbr <= 0.50, 1, 2))      # 0 = <=10%, 1 = 10-50%, 2 = >50%

    segment = np.where(dpd_c <= 0.5,
                np.where(dbr_c <= 1.5, "best", np.where(housing <= 1.5, "better", "good")),
                np.where(dbr_c <= 1.5, np.where(age > 37.5, "bad", "worse"), "worst"))
    decision = np.where(dpd_c <= 0.5,
                np.where(dbr_c <= 1.5, "STP", np.where(housing <= 1.5, "STP", "RTC")),
                np.where(dbr_c <= 1.5, np.where(age > 37.5, "RTC", "STR"), "STR"))
    return int(dpd_c), int(dbr_c), str(segment), str(decision)


SEGMENTS = {
    "best":   ("DPD_30_6M = 0 AND Debt_Burden_Ratio <= 50%", 0.6, 0.6),
    "better": ("DPD_30_6M = 0 AND Debt_Burden_Ratio > 50% AND Housing = Owner", 3.8, 1.5),
    "good":   ("DPD_30_6M = 0 AND Debt_Burden_Ratio > 50% AND Housing = Rent", 9.4, 11.6),
    "bad":    ("DPD_30_6M >= 1 AND Debt_Burden_Ratio <= 50% AND Age > 37.5", 26.7, 28.1),
    "worse":  ("DPD_30_6M >= 1 AND Debt_Burden_Ratio <= 50% AND Age <= 37.5", 48.1, 44.2),
    "worst":  ("DPD_30_6M >= 1 AND Debt_Burden_Ratio > 50%", 89.5, 89.5),
}

---
### 4. Prompt

In [ ]:
VARIABLE_INFO = """
DPD_30_6M - number of tradelines 30+ days past due in the last 6 months. Range: 0, 1 or 2 (or missing); model code 0 = none/missing, 1 = one or more. Strongest risk driver: default rate ~1.3% at code 0 vs ~50.7% at code 1.
Debt_Burden_Ratio - ratio of loan to income. Model code 0 = up to 10%, 1 = 10% to 50%, 2 = above 50%. Higher ratio = higher default: 1.2%, 27.2%, 50.2% by band.
Housing - 1 = Owner, 2 = Renter. Renters default more: ~10.0% owners vs ~20.4% renters.
Age - years, training range 22 to 55. Younger customers default more: ~22% at 37 or below vs ~12% at 38 or above."""

DECISIONS = {"STP": "Straight Through Process (auto approve)",
             "RTC": "Refer to Credit (manual review by a credit manager)",
             "STR": "Straight Through Reject (auto reject)"}

def build_prompt(dpd, dbr, housing, age, dpd_c, dbr_c, segment, decision):
    rule, train_dr, test_dr = SEGMENTS[segment]
    return f"""You are a credit policy assistant. A decision-tree rule engine has ALREADY made the decision below.
Explain in plain language, for a credit underwriter, why the rules led to it. Do not change or question the decision.
Use only the facts given; do not invent numbers.

DECISION: {decision} = {DECISIONS[decision]}
RISK SEGMENT: {segment} - rule applied: {rule}
Historical default rate of this segment: {train_dr}% (train), {test_dr}% (test). Portfolio average: 15.6%.

APPLICANT VALUES:
- DPD_30_6M: {dpd} tradeline(s) 30+ DPD -> code {dpd_c}
- Debt_Burden_Ratio: {dbr:.0%} -> code {dbr_c}
- Housing: {'Owner' if housing == 1 else 'Renter'}
- Age: {age:g}

VARIABLE DEFINITIONS, RANGES AND RELATIONSHIP WITH DEFAULT:
{VARIABLE_INFO}
Write about 200 words: start with the decision, then explain step by step how each variable value led to it and how it relates to default risk, then what the segment default rate means."""

---
### 5. Call the LLM

In [5]:
def ask_llm(prompt):
    r = requests.post(f"{OLLAMA_HOST}/api/chat", timeout=900, json={
        "model": MODEL_NAME, "stream": False,
        "messages": [{"role": "user", "content": prompt}]})
    r.raise_for_status()
    text = r.json()["message"]["content"]
    return re.sub(r"(?s)^.*</think>|<think>.*?</think>", "", text).strip()   # drop the model's thinking block

---
### 6. Inputs

In [ ]:
######## Edit the four values, then run the cell ########
# ---------------- fill in the applicant ----------------
DPD     = 1        # tradelines 30+ DPD in last 6 months. Range: 0, 1, 2 or more (None if missing)
DBR     = 0.30     # debt burden ratio as a decimal. Range: 0 to 2.0 (0.30 = 30%). Bands: <=0.10, 0.10-0.50, >0.50
HOUSING = "Owner"  # "Owner" or "Rent"
AGE     = 34       # years. Range: 18 to 100 (model trained on 22 to 55)
# --------------------------------------------------------

---
### 7. Run

In [6]:
if DPD is not None and DPD < 0:
    raise ValueError("DPD must be 0 or more (or None if missing)")
if not 0 <= DBR <= 2:
    raise ValueError("DBR must be a decimal between 0 and 2.0, e.g. 0.30 for 30%")
if HOUSING not in ("Owner", "Rent"):
    raise ValueError('HOUSING must be "Owner" or "Rent"')
if not 18 <= AGE <= 100:
    raise ValueError("AGE must be between 18 and 100")

housing = 1 if HOUSING == "Owner" else 2
dpd = DPD or 0
dpd_c, dbr_c, segment, decision = decide(dpd, DBR, housing, AGE)
print(f"DECISION: {decision}  (segment: {segment})\n")

print("Asking the LLM to explain (can take a minute) ...\n")
try:
    print(ask_llm(build_prompt(dpd, DBR, housing, AGE, dpd_c, dbr_c, segment, decision)))
except Exception as e:
    print("LLM call failed:", e)
    print("Check that Ollama is running and the model is installed (run 'ollama list' in a terminal).")

DECISION: STR  (segment: worse)

Asking the LLM to explain (can take a minute) ...

**Decision:** The applicant will be rejected automatically (STR = Straight Through Reject).  

---

### Explanation for Credit Underwriter:  
1. **Rule Applied:**  
   - Three criteria must all apply:  
     - **DPD_30_6M ≥ 1**: The applicant has at least one tradeline 30+ days past due in the last 6 months (code = 1).  
     - **Debt_Burden_Ratio ≤ 50%**: Their loan-to-income ratio falls within the "10–50%" band (code = 1).  
     - **Age ≤ 37.5**: The applicant is 34 years old, well below the threshold of 37.5.  

   All three conditions are met, triggering the decision to auto-reject.  

2. **Default Risk Analysis:**  
   - **DPD_30_6M = 1**: Code = 1 for this variable increases default risk from 1.3% (code = 0) to **50.7%**, a substantial multiplier in credit risk.  
   - **Debt_Burden_Ratio = code = 1**: The applicant is at the upper bound of the "10–50%" band, where defaults rise to **27.2%**—sign